# Phân loại ảnh cử chỉ tay — HaGRID (10 lớp)
HOG + SVM/RF/KNN, CNN tự thiết kế, MobileNetV2 / EfficientNet-B0 / ResNet50.

**Cách chạy:** Add Input `hagrid-sample-30k-384p` → Accelerator GPU → Run All.

In [ ]:
%cd /kaggle/working
!pip install -q scikit-image

## 1. Ghi mã nguồn `src/`

In [ ]:
import os
for d in ['src/data', 'src/models', 'outputs', 'data']:
    os.makedirs(d, exist_ok=True)

In [ ]:
%%writefile src/__init__.py
"""Mã nguồn dự án phân loại cử chỉ tay."""

In [ ]:
%%writefile src/benchmark.py
"""Số tham số, kích thước mô hình, thời gian suy luận trên CPU (batch = 1).

    python -m src.benchmark --ckpt outputs/*/best.pt
"""
import argparse
import copy
import io
import time

import numpy as np
import torch


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def model_size_mb(model):
    buf = io.BytesIO()
    torch.save(model.state_dict(), buf)
    return buf.tell() / 1e6


@torch.no_grad()
def cpu_latency_ms(model, img_size=224, n_warmup=10, n_runs=100, threads=None):
    if threads:
        torch.set_num_threads(threads)
    model = copy.deepcopy(model).cpu().eval()
    x = torch.randn(1, 3, img_size, img_size)
    for _ in range(n_warmup):
        model(x)
    times = []
    for _ in range(n_runs):
        t = time.perf_counter()
        model(x)
        times.append((time.perf_counter() - t) * 1000)
    times = np.array(times)
    return {"mean_ms": float(times.mean()), "median_ms": float(np.median(times)),
            "p95_ms": float(np.percentile(times, 95)), "threads": torch.get_num_threads()}


def benchmark_model(model, img_size=224, threads=None):
    return {"params": count_params(model), "size_mb": model_size_mb(model),
            "latency": cpu_latency_ms(model, img_size, threads=threads)}


def main():
    from src.models import load_checkpoint

    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpt", nargs="+", required=True)
    ap.add_argument("--threads", type=int, default=None)
    args = ap.parse_args()

    print(f"{'model':<35}{'params(M)':>10}{'size(MB)':>10}{'CPU ms':>10}")
    for path in args.ckpt:
        model, ckpt = load_checkpoint(path, "cpu")
        b = benchmark_model(model, ckpt["img_size"], args.threads)
        print(f"{path:<35}{b['params'] / 1e6:>10.2f}{b['size_mb']:>10.1f}{b['latency']['mean_ms']:>10.1f}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/compare_results.py
"""Gom tất cả outputs/*/results.json thành bảng so sánh + biểu đồ.

    python -m src.compare_results --out_dir outputs
"""
import argparse
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

from src.utils import load_json


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out_dir", default="outputs")
    args = ap.parse_args()
    out_dir = Path(args.out_dir)

    rows = []
    for f in sorted(out_dir.glob("*/results.json")):
        r = load_json(f)
        rows.append({
            "experiment": r["exp_name"],
            "strategy": r["strategy"],
            "augment": r["augment"],
            "test_acc": r["test"]["accuracy"],
            "test_f1": r["test"]["f1_macro"],
            "test_precision": r["test"]["precision_macro"],
            "test_recall": r["test"]["recall_macro"],
            "custom_acc": r.get("custom", {}).get("accuracy"),
            "custom_f1": r.get("custom", {}).get("f1_macro"),
            "params_M": r["params"] / 1e6 if r.get("params") else None,
            "size_MB": r["size_mb"],
            "cpu_ms": r["latency_ms"],
        })
    if not rows:
        print("Chưa có results.json nào")
        return

    df = pd.DataFrame(rows).sort_values("test_f1", ascending=False)
    df.to_csv(out_dir / "comparison.csv", index=False)
    print(df.round(4).to_string(index=False))

    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    plot_df = df.set_index("experiment")
    plot_df[["test_acc", "test_f1", "custom_acc"]].plot.barh(ax=ax[0])
    ax[0].set_xlim(0, 1)
    ax[0].axvline(0.9, color="red", linestyle="--", linewidth=1)
    ax[0].set_title("Accuracy / F1 (đường đỏ = mục tiêu 90%)")
    plot_df["cpu_ms"].plot.barh(ax=ax[1], color="tab:orange")
    ax[1].axvline(50, color="red", linestyle="--", linewidth=1)
    ax[1].set_title("CPU ms/ảnh (mục tiêu < 50)")
    plt.tight_layout()
    plt.savefig(out_dir / "comparison.png", dpi=150)
    print(f"Đã lưu {out_dir / 'comparison.csv'} và comparison.png")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/config.py
"""Cấu hình dùng chung cho toàn bộ project."""

# 10 lớp cử chỉ lấy từ HaGRID (tên lớp đúng như trong file annotation của HaGRID)
CLASSES = ["call", "dislike", "fist", "like", "ok", "one", "palm", "peace", "rock", "stop"]
NUM_CLASSES = len(CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

VI_NAMES = {
    "call": "Gọi điện", "dislike": "Dislike", "fist": "Nắm tay", "like": "Like",
    "ok": "OK", "one": "Một ngón", "palm": "Xòe tay", "peace": "Chữ V",
    "rock": "Rock", "stop": "Dừng",
}

IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# HOG cho mô hình truyền thống (ảnh nhỏ hơn để vector đặc trưng không quá lớn)
HOG_SIZE = 128

SEED = 42

# Oẳn tù tì: cử chỉ -> nước đi
RPS_MAP = {"fist": "rock", "palm": "paper", "peace": "scissors"}

DL_MODELS = ["cnn", "mobilenet_v2", "efficientnet_b0", "resnet50"]
CLASSICAL_MODELS = ["svm", "rf", "knn"]

In [ ]:
%%writefile src/data/__init__.py
"""Xử lý dữ liệu: cắt ảnh, chia tập, Dataset."""

In [ ]:
%%writefile src/data/crop.py
def square_box(x1, y1, x2, y2, img_w, img_h, margin=0.15):
    """Biến khung bao (pixel) thành khung vuông, mở rộng mỗi phía `margin` và cắt theo biên ảnh."""
    w, h = x2 - x1, y2 - y1
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    side = max(w, h) * (1 + 2 * margin)
    nx1 = int(max(0, cx - side / 2))
    ny1 = int(max(0, cy - side / 2))
    nx2 = int(min(img_w, cx + side / 2))
    ny2 = int(min(img_h, cy + side / 2))
    return nx1, ny1, nx2, ny2

In [ ]:
%%writefile src/data/dataset.py
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset

from src.config import CLASS_TO_IDX
from src.data.transforms import get_eval_transforms, get_train_transforms


class GestureDataset(Dataset):
    def __init__(self, df, root, transform=None):
        self.root = Path(root)
        self.paths = df["path"].tolist()
        self.labels = [CLASS_TO_IDX[l] for l in df["label"]]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.root / self.paths[i]).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, self.labels[i]


def load_split(csv_path, split):
    df = pd.read_csv(csv_path)
    return df[df["split"] == split].reset_index(drop=True)


def make_loader(df, root, transform, batch_size=64, shuffle=False, num_workers=2):
    return DataLoader(
        GestureDataset(df, root, transform), batch_size=batch_size, shuffle=shuffle,
        num_workers=num_workers, pin_memory=torch.cuda.is_available(), drop_last=False,
    )


def make_loaders(data_dir, batch_size=64, num_workers=2, augment=True, img_size=224):
    """Trả về dict {'train', 'val', 'test'} DataLoader đọc từ <data_dir>/splits.csv."""
    csv = Path(data_dir) / "splits.csv"
    train_tf, eval_tf = get_train_transforms(img_size, augment), get_eval_transforms(img_size)
    return {
        "train": make_loader(load_split(csv, "train"), data_dir, train_tf, batch_size, True, num_workers),
        "val": make_loader(load_split(csv, "val"), data_dir, eval_tf, batch_size, False, num_workers),
        "test": make_loader(load_split(csv, "test"), data_dir, eval_tf, batch_size, False, num_workers),
    }


def make_custom_loader(custom_dir, batch_size=64, num_workers=2, img_size=224):
    df = pd.read_csv(Path(custom_dir) / "metadata.csv")
    return make_loader(df, custom_dir, get_eval_transforms(img_size), batch_size, False, num_workers)

In [ ]:
%%writefile src/data/prepare_custom.py
"""Chuẩn bị tập dữ liệu tự thu thập (chỉ dùng để KIỂM THỬ ngoài phân phối).

Đầu vào: data/custom_raw/<class>/*.jpg (ảnh chụp từ webcam/điện thoại)
Đầu ra : data/custom/<class>/*.jpg + data/custom/metadata.csv

    # Ảnh đã cắt sẵn vùng tay:
    python -m src.data.prepare_custom --in_dir data/custom_raw --out_dir data/custom
    # Ảnh nguyên khung -> dùng MediaPipe cắt tay (giống lúc chạy ứng dụng thật):
    python -m src.data.prepare_custom --in_dir data/custom_raw --out_dir data/custom --use_mediapipe
"""
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

from src.config import CLASSES, IMG_SIZE

IMG_EXTS = {".jpg", ".jpeg", ".png"}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--in_dir", default="data/custom_raw")
    ap.add_argument("--out_dir", default="data/custom")
    ap.add_argument("--use_mediapipe", action="store_true")
    ap.add_argument("--margin", type=float, default=0.25)
    args = ap.parse_args()

    detector = None
    if args.use_mediapipe:
        from src.hand_detector import HandDetector  # import muộn để Kaggle không cần cài mediapipe
        detector = HandDetector(static=True, margin=args.margin)

    in_dir, out_dir = Path(args.in_dir), Path(args.out_dir)
    rows, skipped = [], 0
    for cls in CLASSES:
        files = [p for p in (in_dir / cls).glob("*") if p.suffix.lower() in IMG_EXTS]
        (out_dir / cls).mkdir(parents=True, exist_ok=True)
        for p in tqdm(files, desc=cls):
            img = Image.open(p).convert("RGB")
            if detector is not None:
                boxes = detector.detect(np.asarray(img))
                if not boxes:
                    skipped += 1
                    continue
                img = img.crop(boxes[0])
            img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
            rel = f"{cls}/{p.stem}.jpg"
            img.save(out_dir / rel, quality=95)
            rows.append({"path": rel, "label": cls, "user_id": "custom", "source": "custom", "split": "custom"})

    df = pd.DataFrame(rows)
    df.to_csv(out_dir / "metadata.csv", index=False)
    print(df["label"].value_counts())
    print(f"Tổng {len(df)} ảnh, bỏ qua {skipped} ảnh không phát hiện được tay")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/data/prepare_hagrid.py
"""Cắt vùng bàn tay từ HaGRID theo bounding box và lưu thành data/processed/<class>/*.jpg.

Định dạng annotation HaGRID (mỗi lớp 1 file <class>.json):
    { "<image_id>": { "bboxes": [[x, y, w, h], ...],   # tọa độ chuẩn hóa [0, 1], (x, y) là góc trên-trái
                      "labels": ["like", "no_gesture"],
                      "user_id": "..." }, ... }

Ví dụ trên Kaggle (dataset "hagrid-sample-30k-384p"):
    python -m src.data.prepare_hagrid \
        --ann_dir /kaggle/input/hagrid-sample-30k-384p/ann_train_val \
        --img_dir /kaggle/input/hagrid-sample-30k-384p/hagrid_30k \
        --out_dir /kaggle/working/data/processed --max_per_class 2000
"""
import argparse
import json
import random
from pathlib import Path

import pandas as pd
from PIL import Image
from tqdm import tqdm

from src.config import CLASSES, IMG_SIZE, SEED
from src.data.crop import square_box

IMG_EXTS = {".jpg", ".jpeg", ".png"}


def build_image_index(img_root):
    """Map image_id -> đường dẫn file (quét đệ quy, vì mỗi bản HaGRID xếp thư mục hơi khác nhau)."""
    return {p.stem: p for p in Path(img_root).rglob("*") if p.suffix.lower() in IMG_EXTS}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ann_dir", required=True)
    ap.add_argument("--img_dir", required=True)
    ap.add_argument("--out_dir", default="data/processed")
    ap.add_argument("--max_per_class", type=int, default=2000)
    ap.add_argument("--margin", type=float, default=0.15, help="mở rộng biên mỗi phía (10-20%%)")
    ap.add_argument("--classes", nargs="+", default=CLASSES)
    args = ap.parse_args()

    out_dir = Path(args.out_dir)
    print("Đang lập chỉ mục ảnh...")
    index = build_image_index(args.img_dir)
    print(f"Tìm thấy {len(index)} ảnh")

    rows = []
    for cls in args.classes:
        ann_path = Path(args.ann_dir) / f"{cls}.json"
        if not ann_path.exists():
            print(f"[CẢNH BÁO] Không có {ann_path}, bỏ qua lớp {cls}")
            continue
        with open(ann_path, encoding="utf-8") as f:
            anns = json.load(f)

        ids = list(anns.keys())
        random.Random(SEED).shuffle(ids)
        (out_dir / cls).mkdir(parents=True, exist_ok=True)

        n = 0
        for img_id in tqdm(ids, desc=cls):
            if n >= args.max_per_class:
                break
            if img_id not in index:
                continue
            ann = anns[img_id]
            if cls not in ann["labels"]:
                continue
            bx, by, bw, bh = ann["bboxes"][ann["labels"].index(cls)]

            img = Image.open(index[img_id]).convert("RGB")
            W, H = img.size
            box = square_box(bx * W, by * H, (bx + bw) * W, (by + bh) * H, W, H, args.margin)
            if box[2] - box[0] < 10 or box[3] - box[1] < 10:
                continue
            crop = img.crop(box).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)

            rel = f"{cls}/{img_id}.jpg"
            crop.save(out_dir / rel, quality=95)
            rows.append({"path": rel, "label": cls, "user_id": ann.get("user_id", img_id), "source": "hagrid"})
            n += 1
        print(f"{cls}: {n} ảnh")

    df = pd.DataFrame(rows)
    df.to_csv(out_dir / "metadata.csv", index=False)
    print(f"Đã lưu {len(df)} ảnh, {df['user_id'].nunique()} người -> {out_dir / 'metadata.csv'}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/data/split.py
"""Chia train/val/test = 70/15/15, phân tầng theo lớp, KHÔNG để cùng một người nằm ở 2 tập.

    python -m src.data.split --data_dir /kaggle/working/data/processed
"""
import argparse
from pathlib import Path

import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

from src.config import SEED


def group_stratified_split(df, val_ratio=0.15, test_ratio=0.15, seed=SEED):
    df = df.reset_index(drop=True).copy()

    # Bước 1: tách test (~1/7 ≈ 15%)
    sgkf = StratifiedGroupKFold(n_splits=round(1 / test_ratio), shuffle=True, random_state=seed)
    trval_idx, test_idx = next(sgkf.split(df, df["label"], df["user_id"]))
    df.loc[test_idx, "split"] = "test"

    # Bước 2: tách val từ phần còn lại (~15% tổng)
    trval = df.iloc[trval_idx]
    sgkf = StratifiedGroupKFold(n_splits=round((1 - test_ratio) / val_ratio), shuffle=True, random_state=seed)
    tr_idx, va_idx = next(sgkf.split(trval, trval["label"], trval["user_id"]))
    df.loc[trval.index[tr_idx], "split"] = "train"
    df.loc[trval.index[va_idx], "split"] = "val"
    return df


def check_no_user_leak(df):
    users = {s: set(df.loc[df.split == s, "user_id"]) for s in ["train", "val", "test"]}
    assert not users["train"] & users["test"], "Rò rỉ người dùng giữa train và test!"
    assert not users["train"] & users["val"], "Rò rỉ người dùng giữa train và val!"
    assert not users["val"] & users["test"], "Rò rỉ người dùng giữa val và test!"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", default="data/processed")
    args = ap.parse_args()

    data_dir = Path(args.data_dir)
    df = group_stratified_split(pd.read_csv(data_dir / "metadata.csv"))
    check_no_user_leak(df)
    df.to_csv(data_dir / "splits.csv", index=False)

    print(pd.crosstab(df["label"], df["split"], margins=True))
    print((df["split"].value_counts(normalize=True) * 100).round(1))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/data/transforms.py
from torchvision import transforms as T

from src.config import IMAGENET_MEAN, IMAGENET_STD, IMG_SIZE


def get_eval_transforms(img_size=IMG_SIZE):
    return T.Compose([
        T.Resize((img_size, img_size)),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


def get_train_transforms(img_size=IMG_SIZE, augment=True):
    """augment=False dùng cho thực nghiệm ablation (không tăng cường dữ liệu)."""
    if not augment:
        return get_eval_transforms(img_size)
    return T.Compose([
        T.RandomResizedCrop(img_size, scale=(0.75, 1.0), ratio=(0.9, 1.1)),
        T.RandomHorizontalFlip(),
        T.RandomRotation(15),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

In [ ]:
%%writefile src/evaluate.py
"""Độ đo đánh giá + vẽ biểu đồ. Có thể chạy riêng để đánh giá lại một checkpoint:

    python -m src.evaluate --ckpt outputs/mobilenet_v2_finetune/best.pt \
        --data_dir data/processed --custom_dir data/custom
"""
import argparse
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             precision_recall_fscore_support)

from src.utils import get_device, save_json


@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    ys, probs = [], []
    for x, y in loader:
        logits = model(x.to(device, non_blocking=True))
        probs.append(torch.softmax(logits.float(), 1).cpu())
        ys.append(y)
    probs = torch.cat(probs).numpy()
    return torch.cat(ys).numpy(), probs.argmax(1), probs


def compute_metrics(y_true, y_pred, classes):
    labels = list(range(len(classes)))
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": p,
        "recall_macro": r,
        "f1_macro": f1,
        "report": classification_report(y_true, y_pred, labels=labels, target_names=classes,
                                        output_dict=True, zero_division=0),
    }


def plot_confusion_matrix(y_true, y_pred, classes, path, title="Confusion matrix", normalize=True):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(classes))))
    if normalize:
        cm = cm.astype(float) / np.maximum(cm.sum(1, keepdims=True), 1)
    plt.figure(figsize=(9, 7.5))
    sns.heatmap(cm, annot=True, fmt=".2f" if normalize else "d", cmap="Blues",
                xticklabels=classes, yticklabels=classes, vmin=0, vmax=1 if normalize else None)
    plt.xlabel("Dự đoán")
    plt.ylabel("Thực tế")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()


def plot_history(history, path):
    epochs = [h["epoch"] for h in history]
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
    ax[0].plot(epochs, [h["train_loss"] for h in history], label="train")
    ax[0].plot(epochs, [h["val_loss"] for h in history], label="val")
    ax[0].set_title("Loss")
    ax[1].plot(epochs, [h["train_acc"] for h in history], label="train acc")
    ax[1].plot(epochs, [h["val_acc"] for h in history], label="val acc")
    ax[1].plot(epochs, [h["val_f1"] for h in history], label="val F1", linestyle="--")
    ax[1].set_title("Accuracy / F1")
    for a in ax:
        a.set_xlabel("epoch")
        a.legend()
        a.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()


def evaluate_and_save(model, loader, classes, device, out_dir, tag):
    """Dự đoán, tính độ đo, vẽ ma trận nhầm lẫn; trả về dict độ đo."""
    y_true, y_pred, _ = predict(model, loader, device)
    m = compute_metrics(y_true, y_pred, classes)
    plot_confusion_matrix(y_true, y_pred, classes, Path(out_dir) / f"cm_{tag}.png",
                          title=f"{tag}: acc={m['accuracy']:.3f}, F1={m['f1_macro']:.3f}")
    print(f"[{tag}] acc={m['accuracy']:.4f}  P={m['precision_macro']:.4f}  "
          f"R={m['recall_macro']:.4f}  F1={m['f1_macro']:.4f}")
    return m


def main():
    from src.data.dataset import make_custom_loader, make_loaders
    from src.models import load_checkpoint

    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--data_dir", default="data/processed")
    ap.add_argument("--custom_dir", default=None)
    ap.add_argument("--batch_size", type=int, default=64)
    ap.add_argument("--num_workers", type=int, default=2)
    args = ap.parse_args()

    device = get_device()
    model, ckpt = load_checkpoint(args.ckpt, device)
    out_dir = Path(args.ckpt).parent
    results = {}
    loaders = make_loaders(args.data_dir, args.batch_size, args.num_workers, img_size=ckpt["img_size"])
    results["test"] = evaluate_and_save(model, loaders["test"], ckpt["classes"], device, out_dir, "test")
    if args.custom_dir:
        cl = make_custom_loader(args.custom_dir, args.batch_size, args.num_workers, ckpt["img_size"])
        results["custom"] = evaluate_and_save(model, cl, ckpt["classes"], device, out_dir, "custom")
    save_json(results, out_dir / "eval.json")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/gradcam.py
"""Grad-CAM: kiểm tra mô hình có nhìn vào vùng bàn tay hay không.

    python -m src.gradcam --ckpt outputs/mobilenet_v2_finetune/best.pt --data_dir data/processed --n 16
"""
import argparse
import math
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn.functional as F


class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model.eval()
        self.acts = None
        self.grads = None
        self.handle = target_layer.register_forward_hook(self._hook)

    def _hook(self, module, inp, out):
        self.acts = out
        if out.requires_grad:
            out.register_hook(lambda g: setattr(self, "grads", g))

    def __call__(self, x, class_idx=None):
        """x: tensor (1, 3, H, W) đã chuẩn hóa. Trả về (cam HxW trong [0,1], class_idx, probs)."""
        self.model.zero_grad(set_to_none=True)
        with torch.enable_grad():
            logits = self.model(x)
            if class_idx is None:
                class_idx = int(logits.argmax(1).item())
            logits[0, class_idx].backward()
        weights = self.grads.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * self.acts).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[2:], mode="bilinear", align_corners=False)[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        probs = torch.softmax(logits, 1)[0]
        return cam.detach().cpu().numpy(), class_idx, probs.detach().cpu().numpy()

    def remove(self):
        self.handle.remove()


def overlay_cam(rgb, cam, alpha=0.45):
    """rgb: uint8 HxWx3 (RGB). Trả về ảnh RGB đã chồng heatmap."""
    heat = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)
    heat = cv2.resize(heat, (rgb.shape[1], rgb.shape[0]))
    return np.uint8(alpha * heat + (1 - alpha) * rgb)


def main():
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from PIL import Image

    from src.data.dataset import load_split
    from src.data.transforms import get_eval_transforms
    from src.models import get_gradcam_layer, load_checkpoint

    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--data_dir", default="data/processed")
    ap.add_argument("--csv", default=None, help="mặc định <data_dir>/splits.csv; dùng data/custom/metadata.csv cho tập tự thu")
    ap.add_argument("--split", default="test")
    ap.add_argument("--n", type=int, default=16)
    ap.add_argument("--out", default=None)
    args = ap.parse_args()

    model, ckpt = load_checkpoint(args.ckpt, "cpu")
    classes, size = ckpt["classes"], ckpt["img_size"]
    cam_fn = GradCAM(model, get_gradcam_layer(model, ckpt["model_name"]))
    tf = get_eval_transforms(size)

    df = load_split(args.csv or Path(args.data_dir) / "splits.csv", args.split)
    df = df.sample(min(args.n, len(df)), random_state=0)

    cols = 4
    rows = math.ceil(len(df) / cols)
    fig, axes = plt.subplots(rows, cols * 2, figsize=(cols * 5, rows * 2.6))
    axes = np.atleast_2d(axes)
    for ax in axes.ravel():
        ax.axis("off")
    for i, (_, r) in enumerate(df.iterrows()):
        img = Image.open(Path(args.data_dir) / r["path"]).convert("RGB").resize((size, size))
        cam, pred, probs = cam_fn(tf(img).unsqueeze(0))
        rgb = np.asarray(img)
        ok = classes[pred] == r["label"]
        a1, a2 = axes[i // cols, (i % cols) * 2], axes[i // cols, (i % cols) * 2 + 1]
        a1.imshow(rgb)
        a1.set_title(f"GT: {r['label']}", fontsize=9)
        a2.imshow(overlay_cam(rgb, cam))
        a2.set_title(f"Pred: {classes[pred]} ({probs[pred]:.2f})", fontsize=9, color="green" if ok else "red")
    plt.tight_layout()
    out = args.out or Path(args.ckpt).parent / f"gradcam_{args.split}.png"
    plt.savefig(out, dpi=130)
    print(f"Đã lưu {out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/hand_detector.py
"""MediaPipe Hands: CHỈ dùng để định vị bàn tay, việc phân loại do mô hình của mình làm."""
import mediapipe as mp

from src.data.crop import square_box


class HandDetector:
    def __init__(self, max_hands=1, min_conf=0.5, static=False, margin=0.25):
        self.hands = mp.solutions.hands.Hands(
            static_image_mode=static, max_num_hands=max_hands,
            min_detection_confidence=min_conf, min_tracking_confidence=0.5,
        )
        self.margin = margin

    def detect(self, rgb):
        """rgb: ndarray HxWx3 uint8 (RGB). Trả về list khung (x1, y1, x2, y2) vuông, đã mở rộng biên."""
        h, w = rgb.shape[:2]
        res = self.hands.process(rgb)
        boxes = []
        for lm in res.multi_hand_landmarks or []:
            xs = [p.x * w for p in lm.landmark]
            ys = [p.y * h for p in lm.landmark]
            boxes.append(square_box(min(xs), min(ys), max(xs), max(ys), w, h, self.margin))
        return boxes

    def close(self):
        self.hands.close()

In [ ]:
%%writefile src/models/__init__.py
import torch

from src.models.cnn import SimpleCNN
from src.models.transfer import build_transfer


def build_model(name, num_classes, pretrained=True):
    if name == "cnn":
        return SimpleCNN(num_classes)
    return build_transfer(name, num_classes, pretrained=pretrained)


def get_gradcam_layer(model, name):
    """Tầng tích chập cuối dùng cho Grad-CAM."""
    if name == "resnet50":
        return model.layer4[-1]
    return model.features[-1]  # cnn, mobilenet_v2, efficientnet_b0


def load_checkpoint(path, device="cpu"):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model = build_model(ckpt["model_name"], len(ckpt["classes"]), pretrained=False)
    model.load_state_dict(ckpt["state_dict"])
    return model.to(device).eval(), ckpt

In [ ]:
%%writefile src/models/classical.py
"""Mô hình cơ sở: HOG + (SVM | Random Forest | KNN)."""
import numpy as np
from joblib import Parallel, delayed
from PIL import Image
from skimage.feature import hog
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from tqdm import tqdm

from src.config import HOG_SIZE, SEED

HOG_PARAMS = dict(orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2),
                  block_norm="L2-Hys", transform_sqrt=True, feature_vector=True)


def hog_from_pil(img, size=HOG_SIZE):
    """Ảnh xám, chuẩn hóa về [0, 1], resize về size x size -> vector HOG (8100 chiều với 128x128)."""
    gray = np.asarray(img.convert("L").resize((size, size)), dtype=np.float32) / 255.0
    return hog(gray, **HOG_PARAMS).astype(np.float32)


def hog_from_path(path, size=HOG_SIZE):
    return hog_from_pil(Image.open(path), size)


def extract_hog_features(paths, n_jobs=-1):
    feats = Parallel(n_jobs=n_jobs, batch_size=64)(delayed(hog_from_path)(p) for p in tqdm(paths, desc="HOG"))
    return np.stack(feats)


def build_pipeline(name, pca_components=200):
    steps = [("scaler", StandardScaler())]
    if pca_components:
        steps.append(("pca", PCA(n_components=pca_components, random_state=SEED)))
    if name == "svm":
        clf = SVC(kernel="rbf", random_state=SEED)
    elif name == "rf":
        clf = RandomForestClassifier(random_state=SEED, n_jobs=1)
    elif name == "knn":
        clf = KNeighborsClassifier(n_jobs=1)
    else:
        raise ValueError(name)
    steps.append(("clf", clf))
    return Pipeline(steps)


PARAM_GRIDS = {
    "svm": {"clf__C": [1, 10, 100], "clf__gamma": ["scale", 1e-3, 1e-2]},
    "rf": {"clf__n_estimators": [200, 500], "clf__max_depth": [None, 30], "clf__max_features": ["sqrt", "log2"]},
    "knn": {"clf__n_neighbors": [3, 5, 7, 11], "clf__weights": ["uniform", "distance"],
            "clf__metric": ["euclidean", "cosine"]},
}

In [ ]:
%%writefile src/models/cnn.py
import torch.nn as nn


def conv_block(cin, cout):
    """Conv - BN - ReLU (x2) - MaxPool: giảm kích thước không gian đi 2 lần."""
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False),
        nn.BatchNorm2d(cout),
        nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1, bias=False),
        nn.BatchNorm2d(cout),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2),
    )


class SimpleCNN(nn.Module):
    """CNN tự thiết kế, huấn luyện từ đầu. 224 -> 112 -> 56 -> 28 -> 14."""

    def __init__(self, num_classes, channels=(32, 64, 128, 256), dropout=0.4):
        super().__init__()
        layers, cin = [], 3
        for c in channels:
            layers.append(conv_block(cin, c))
            cin = c
        self.features = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(cin, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout / 2),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))

In [ ]:
%%writefile src/models/transfer.py
"""Học chuyển giao: MobileNetV2, EfficientNet-B0, ResNet50 (pretrained ImageNet, torchvision)."""
import torch.nn as nn
from torch.nn.modules.batchnorm import _BatchNorm
from torchvision import models

HEAD_ATTR = {"mobilenet_v2": "classifier", "efficientnet_b0": "classifier", "resnet50": "fc"}

# Số khối cuối được mở băng khi fine-tuning
DEFAULT_UNFREEZE = {"mobilenet_v2": 5, "efficientnet_b0": 3, "resnet50": 1}


def build_transfer(name, num_classes, pretrained=True, dropout=0.2):
    if name == "mobilenet_v2":
        m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None)
        m.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(m.last_channel, num_classes))
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
        m.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(m.classifier[1].in_features, num_classes))
    elif name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
        m.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(m.fc.in_features, num_classes))
    else:
        raise ValueError(f"Mô hình không hỗ trợ: {name}")
    return m


def freeze_backbone(model, name):
    """Feature extraction: chỉ huấn luyện lớp phân loại cuối."""
    for p in model.parameters():
        p.requires_grad = False
    for p in getattr(model, HEAD_ATTR[name]).parameters():
        p.requires_grad = True


def unfreeze_last_blocks(model, name, n_blocks=None):
    """Fine-tuning: mở băng n khối cuối của mạng nền."""
    n_blocks = n_blocks or DEFAULT_UNFREEZE[name]
    if name in ("mobilenet_v2", "efficientnet_b0"):
        blocks = list(model.features.children())[-n_blocks:]
    else:  # resnet50
        blocks = [model.layer1, model.layer2, model.layer3, model.layer4][-n_blocks:]
    for b in blocks:
        for p in b.parameters():
            p.requires_grad = True


def freeze_bn_stats(model):
    """Giữ BatchNorm của phần bị đóng băng ở chế độ eval để không làm trôi thống kê ImageNet."""
    for m in model.modules():
        if isinstance(m, _BatchNorm) and not any(p.requires_grad for p in m.parameters()):
            m.eval()

In [ ]:
%%writefile src/train_classical.py
"""Huấn luyện HOG + SVM / Random Forest / KNN với Grid Search + kiểm định chéo theo nhóm người.

    python -m src.train_classical --model svm rf knn --data_dir data/processed --custom_dir data/custom
"""
import argparse
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold

from src.config import CLASS_TO_IDX, CLASSES, CLASSICAL_MODELS, SEED
from src.evaluate import compute_metrics, plot_confusion_matrix
from src.models.classical import PARAM_GRIDS, build_pipeline, extract_hog_features, hog_from_pil
from src.utils import save_json


def load_features(df, root, cache_path, n_jobs):
    if Path(cache_path).exists():
        return np.load(cache_path)
    X = extract_hog_features([Path(root) / p for p in df["path"]], n_jobs)
    np.save(cache_path, X)
    return X


def measure_latency_ms(pipe, paths, n=100):
    """Thời gian 1 ảnh = đọc ảnh + HOG + dự đoán (cùng điều kiện như mô hình DL: batch 1, CPU)."""
    paths = list(paths)[:n]
    times = []
    for p in paths:
        img = Image.open(p)
        img.load()
        t = time.perf_counter()
        pipe.predict(hog_from_pil(img)[None])
        times.append((time.perf_counter() - t) * 1000)
    return float(np.mean(times))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", nargs="+", default=CLASSICAL_MODELS, choices=CLASSICAL_MODELS)
    ap.add_argument("--data_dir", default="data/processed")
    ap.add_argument("--custom_dir", default=None)
    ap.add_argument("--out_dir", default="outputs")
    ap.add_argument("--pca", type=int, default=200, help="số thành phần PCA (0 = không dùng)")
    ap.add_argument("--cv", type=int, default=3)
    ap.add_argument("--n_jobs", type=int, default=-1)
    args = ap.parse_args()

    out_dir = Path(args.out_dir)
    cache_dir = out_dir / "hog_cache"
    cache_dir.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(Path(args.data_dir) / "splits.csv")
    # Grid Search dùng CV trên train+val (val đóng vai trò fold), test giữ nguyên để báo cáo
    trval = df[df.split.isin(["train", "val"])].reset_index(drop=True)
    test = df[df.split == "test"].reset_index(drop=True)
    X_tr = load_features(trval, args.data_dir, cache_dir / "trainval.npy", args.n_jobs)
    X_te = load_features(test, args.data_dir, cache_dir / "test.npy", args.n_jobs)
    y_tr = trval["label"].map(CLASS_TO_IDX).values
    y_te = test["label"].map(CLASS_TO_IDX).values
    print(f"HOG: train+val {X_tr.shape}, test {X_te.shape}")

    custom = None
    if args.custom_dir:
        cdf = pd.read_csv(Path(args.custom_dir) / "metadata.csv")
        X_cu = load_features(cdf, args.custom_dir, cache_dir / "custom.npy", args.n_jobs)
        custom = (X_cu, cdf["label"].map(CLASS_TO_IDX).values)

    for name in args.model:
        exp_dir = out_dir / f"hog_{name}"
        exp_dir.mkdir(parents=True, exist_ok=True)
        print(f"\n===== HOG + {name.upper()} =====")

        cv = StratifiedGroupKFold(n_splits=args.cv, shuffle=True, random_state=SEED)
        gs = GridSearchCV(build_pipeline(name, args.pca), PARAM_GRIDS[name], cv=cv,
                          scoring="f1_macro", n_jobs=args.n_jobs, verbose=1, refit=True)
        t0 = time.time()
        gs.fit(X_tr, y_tr, groups=trval["user_id"].values)
        train_time = time.time() - t0
        print(f"Best params: {gs.best_params_} | CV F1 = {gs.best_score_:.4f} | {train_time:.0f}s")

        pipe = gs.best_estimator_
        results = {"exp_name": f"hog_{name}", "model": f"hog_{name}", "strategy": "classical",
                   "augment": False, "best_params": gs.best_params_, "cv_f1": gs.best_score_,
                   "train_time_s": train_time}

        y_pred = pipe.predict(X_te)
        results["test"] = compute_metrics(y_te, y_pred, CLASSES)
        plot_confusion_matrix(y_te, y_pred, CLASSES, exp_dir / "cm_test.png",
                              title=f"HOG+{name.upper()} test: acc={results['test']['accuracy']:.3f}")
        print(f"[test] acc={results['test']['accuracy']:.4f} F1={results['test']['f1_macro']:.4f}")

        if custom is not None:
            y_pred_c = pipe.predict(custom[0])
            results["custom"] = compute_metrics(custom[1], y_pred_c, CLASSES)
            plot_confusion_matrix(custom[1], y_pred_c, CLASSES, exp_dir / "cm_custom.png",
                                  title=f"HOG+{name.upper()} custom")
            print(f"[custom] acc={results['custom']['accuracy']:.4f} F1={results['custom']['f1_macro']:.4f}")

        model_path = exp_dir / "model.joblib"
        joblib.dump(pipe, model_path)
        results["size_mb"] = model_path.stat().st_size / 1e6
        results["params"] = None
        results["latency_ms"] = measure_latency_ms(pipe, (Path(args.data_dir) / p for p in test["path"]))
        print(f"Kích thước {results['size_mb']:.1f}MB | CPU {results['latency_ms']:.1f} ms/ảnh")
        save_json(results, exp_dir / "results.json")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/train_dl.py
"""Huấn luyện CNN tự thiết kế / MobileNetV2 / EfficientNet-B0 / ResNet50.

Chiến lược (--strategy):
    scratch          : huấn luyện toàn bộ từ đầu (mặc định cho cnn)
    feature_extract  : đóng băng mạng nền, chỉ học lớp phân loại
    finetune         : học lớp phân loại --head_epochs epoch, sau đó mở băng các khối cuối (mặc định cho transfer)

Ví dụ:
    python -m src.train_dl --model cnn --epochs 40 --lr 1e-3
    python -m src.train_dl --model mobilenet_v2 --strategy finetune
    python -m src.train_dl --model mobilenet_v2 --strategy finetune --no_aug --exp_name mbv2_noaug   # ablation
"""
import argparse
from pathlib import Path

import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

from src.benchmark import benchmark_model
from src.config import CLASSES, DL_MODELS, IMG_SIZE, NUM_CLASSES, SEED
from src.data.dataset import make_custom_loader, make_loaders
from src.evaluate import evaluate_and_save, plot_history
from src.models import build_model, load_checkpoint
from src.models.transfer import freeze_backbone, freeze_bn_stats, unfreeze_last_blocks
from src.utils import get_device, save_json, seed_everything


def parse_args():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True, choices=DL_MODELS)
    ap.add_argument("--strategy", choices=["scratch", "feature_extract", "finetune"], default=None)
    ap.add_argument("--data_dir", default="data/processed")
    ap.add_argument("--custom_dir", default=None, help="tập tự thu thập để đánh giá tổng quát hóa")
    ap.add_argument("--out_dir", default="outputs")
    ap.add_argument("--exp_name", default=None)
    ap.add_argument("--epochs", type=int, default=30)
    ap.add_argument("--head_epochs", type=int, default=5, help="số epoch chỉ học head trước khi fine-tune")
    ap.add_argument("--unfreeze_blocks", type=int, default=None)
    ap.add_argument("--batch_size", type=int, default=64)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--ft_lr", type=float, default=1e-4, help="learning rate giai đoạn fine-tune")
    ap.add_argument("--weight_decay", type=float, default=1e-4)
    ap.add_argument("--optimizer", choices=["adam", "adamw"], default="adamw")
    ap.add_argument("--scheduler", choices=["cosine", "plateau"], default="cosine")
    ap.add_argument("--patience", type=int, default=7, help="early stopping")
    ap.add_argument("--label_smoothing", type=float, default=0.0)
    ap.add_argument("--no_aug", action="store_true", help="tắt tăng cường dữ liệu (ablation)")
    ap.add_argument("--img_size", type=int, default=IMG_SIZE)
    ap.add_argument("--num_workers", type=int, default=2)
    ap.add_argument("--seed", type=int, default=SEED)
    args = ap.parse_args()
    if args.strategy is None:
        args.strategy = "scratch" if args.model == "cnn" else "finetune"
    if args.model == "cnn" and args.strategy != "scratch":
        ap.error("CNN tự thiết kế chỉ dùng --strategy scratch")
    if args.exp_name is None:
        args.exp_name = f"{args.model}_{args.strategy}" + ("_noaug" if args.no_aug else "")
    return args


def make_optimizer(model, args, lr):
    params = [p for p in model.parameters() if p.requires_grad]
    opt_cls = torch.optim.Adam if args.optimizer == "adam" else torch.optim.AdamW
    return opt_cls(params, lr=lr, weight_decay=args.weight_decay)


def make_scheduler(optimizer, args, n_epochs):
    if args.scheduler == "cosine":
        return CosineAnnealingLR(optimizer, T_max=max(1, n_epochs))
    return ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)


def run_epoch(model, loader, criterion, device, optimizer=None, scaler=None):
    train = optimizer is not None
    model.train(train)
    if train:
        freeze_bn_stats(model)
    total_loss, preds, targets = 0.0, [], []
    for x, y in tqdm(loader, leave=False, desc="train" if train else "val"):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.set_grad_enabled(train), torch.autocast(device.type, enabled=device.type == "cuda"):
            logits = model(x)
            loss = criterion(logits, y)
        if train:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        total_loss += loss.item() * x.size(0)
        preds.append(logits.argmax(1).cpu())
        targets.append(y.cpu())
    preds, targets = torch.cat(preds).numpy(), torch.cat(targets).numpy()
    return (total_loss / len(loader.dataset), accuracy_score(targets, preds),
            f1_score(targets, preds, average="macro"))


def main():
    args = parse_args()
    seed_everything(args.seed)
    device = get_device()
    exp_dir = Path(args.out_dir) / args.exp_name
    exp_dir.mkdir(parents=True, exist_ok=True)
    writer = SummaryWriter(exp_dir / "tb")
    print(f"Thực nghiệm: {args.exp_name} | thiết bị: {device}")

    loaders = make_loaders(args.data_dir, args.batch_size, args.num_workers, not args.no_aug, args.img_size)
    print({k: len(v.dataset) for k, v in loaders.items()})

    model = build_model(args.model, NUM_CLASSES, pretrained=args.strategy != "scratch").to(device)
    if args.strategy in ("feature_extract", "finetune"):
        freeze_backbone(model, args.model)

    criterion = nn.CrossEntropyLoss(label_smoothing=args.label_smoothing)
    scaler = torch.amp.GradScaler(enabled=device.type == "cuda")
    first_stage_epochs = args.head_epochs if args.strategy == "finetune" else args.epochs
    optimizer = make_optimizer(model, args, args.lr)
    scheduler = make_scheduler(optimizer, args, first_stage_epochs)

    best_f1, best_epoch, bad_epochs, history = -1.0, 0, 0, []
    ckpt_path = exp_dir / "best.pt"
    for epoch in range(1, args.epochs + 1):
        in_head_stage = args.strategy == "finetune" and epoch <= args.head_epochs
        if args.strategy == "finetune" and epoch == args.head_epochs + 1:
            unfreeze_last_blocks(model, args.model, args.unfreeze_blocks)
            optimizer = make_optimizer(model, args, args.ft_lr)
            scheduler = make_scheduler(optimizer, args, args.epochs - args.head_epochs)
            bad_epochs = 0
            n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"==> Bắt đầu fine-tune, số tham số huấn luyện: {n_train / 1e6:.2f}M")

        tr_loss, tr_acc, tr_f1 = run_epoch(model, loaders["train"], criterion, device, optimizer, scaler)
        va_loss, va_acc, va_f1 = run_epoch(model, loaders["val"], criterion, device)
        lr = optimizer.param_groups[0]["lr"]
        if isinstance(scheduler, ReduceLROnPlateau):
            scheduler.step(va_f1)
        else:
            scheduler.step()

        history.append(dict(epoch=epoch, train_loss=tr_loss, train_acc=tr_acc, train_f1=tr_f1,
                            val_loss=va_loss, val_acc=va_acc, val_f1=va_f1, lr=lr))
        writer.add_scalars("loss", {"train": tr_loss, "val": va_loss}, epoch)
        writer.add_scalars("acc", {"train": tr_acc, "val": va_acc}, epoch)
        writer.add_scalar("val_f1", va_f1, epoch)
        writer.add_scalar("lr", lr, epoch)
        print(f"Epoch {epoch:02d} | lr {lr:.1e} | train loss {tr_loss:.4f} acc {tr_acc:.4f} "
              f"| val loss {va_loss:.4f} acc {va_acc:.4f} F1 {va_f1:.4f}")

        if va_f1 > best_f1:
            best_f1, best_epoch, bad_epochs = va_f1, epoch, 0
            torch.save({"model_name": args.model, "state_dict": model.state_dict(), "classes": CLASSES,
                        "img_size": args.img_size, "epoch": epoch, "val_f1": va_f1, "args": vars(args)}, ckpt_path)
        elif not in_head_stage:
            bad_epochs += 1
            if bad_epochs >= args.patience:
                print(f"Early stopping tại epoch {epoch}")
                break
    writer.close()
    plot_history(history, exp_dir / "history.png")

    # ---- Đánh giá mô hình tốt nhất ----
    model, _ = load_checkpoint(ckpt_path, device)
    results = {"exp_name": args.exp_name, "model": args.model, "strategy": args.strategy,
               "augment": not args.no_aug, "best_epoch": best_epoch, "val_f1": best_f1, "history": history}
    results["test"] = evaluate_and_save(model, loaders["test"], CLASSES, device, exp_dir, "test")
    if args.custom_dir:
        cl = make_custom_loader(args.custom_dir, args.batch_size, args.num_workers, args.img_size)
        results["custom"] = evaluate_and_save(model, cl, CLASSES, device, exp_dir, "custom")

    b = benchmark_model(model, args.img_size)
    results.update(params=b["params"], size_mb=b["size_mb"], latency_ms=b["latency"]["mean_ms"])
    print(f"Tham số: {b['params'] / 1e6:.2f}M | {b['size_mb']:.1f}MB | CPU {b['latency']['mean_ms']:.1f} ms/ảnh")
    save_json(results, exp_dir / "results.json")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/utils.py
import json
import os
import random
from pathlib import Path

import numpy as np
import torch


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _to_serializable(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    raise TypeError(f"Không serialize được kiểu {type(o)}")


def save_json(obj, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=_to_serializable)


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

## 2. Chuẩn bị dữ liệu
Tự dò thư mục annotation (`ann_train_val`) và thư mục ảnh trong input.

In [ ]:
from pathlib import Path
INPUT = Path('/kaggle/input/hagrid-sample-30k-384p')
ANN_DIR = next(p.parent for p in INPUT.rglob('like.json'))
IMG_DIR = INPUT
print('ANN_DIR =', ANN_DIR)

In [ ]:
!python -m src.data.prepare_hagrid --ann_dir "{ANN_DIR}" --img_dir "{IMG_DIR}" --out_dir data/processed --max_per_class 2000
!python -m src.data.split --data_dir data/processed

## 3. Học máy truyền thống (HOG + SVM / RF / KNN)

In [ ]:
!python -m src.train_classical --model svm rf knn

## 4. CNN tự thiết kế

In [ ]:
!python -m src.train_dl --model cnn --epochs 40

## 5. Học chuyển giao

In [ ]:
!python -m src.train_dl --model mobilenet_v2 --strategy finetune
!python -m src.train_dl --model efficientnet_b0 --strategy finetune
!python -m src.train_dl --model resnet50 --strategy finetune

## 6. So sánh kết quả

In [ ]:
!python -m src.compare_results
from IPython.display import Image, display
display(Image('outputs/comparison.png'))

## 7. Grad-CAM (MobileNetV2)

In [ ]:
!python -m src.gradcam --ckpt outputs/mobilenet_v2_finetune/best.pt --n 16
import glob
for f in glob.glob('outputs/mobilenet_v2_finetune/*gradcam*.png'):
    display(Image(f))

## 8. Đóng gói kết quả để tải về

In [ ]:
!cd outputs && zip -qr ../outputs.zip . -x 'hog_cache/*' '*/tb/*' && ls -lh ../outputs.zip